# Lesson 01 - AI एजंट्स परिचय

**AI एजंट्स फॉर बिगिनर्स** कोर्समधील पहिल्या धड्यात आपले स्वागत आहे!

एखादा **AI एजंट** म्हणजे एक प्रोग्राम जो मोठ्या भाषिक मॉडेलचा (LLM) वापर त्याचा विचार करण्याचा इंजिन म्हणून करतो आणि प्रत्यक्ष जगात *क्रियाकलाप* करू शकतो — API कॉल करणे, डेटाबेसमध्ये क्वेरी करणे, किंवा कोड चालवणे — याचा उद्देश वापरकर्त्याच्या वतीने एखादे लक्ष्य साध्य करणे असते.

या नोटबुकमध्ये तुम्ही तुमचा पहिला एजंट तयार कराल: एक **ट्रॅव्हल एजंट** जो सुट्टीसाठी गंतव्य ठिकाणांची शिफारस करतो. याच मार्गदर्शकात तुम्हाला शिकायला मिळेल कसे:

1. **मायक्रोसॉफ्ट एजंट फ्रेमवर्क** वापरून Azure AI Foundry Agent Service शी कनेक्ट करायचे.
2. एजंटला एक **टूल** द्यायचे — एक सोपी Python फंक्शन जी तो कॉल करू शकतो.
3. एजंट चालवायचा आणि त्याचा प्रतिसाद तपासायचा.
4. एजंटचा प्रतिसाद टोकन-बाय-टोकन स्ट्रीम करायचा.


## सेटअप

हा नोटबुक चालवण्यापूर्वी, कृपया खात्री करा की आपल्याकडे:

1. **एखादा Azure AI Foundry प्रकल्प** आहे ज्यामध्ये एक तैनात केलेला चॅट मॉडेल आहे (उदा. `gpt-4o-mini`).
2. **Azure CLI वापरून लॉगिन केलेले आहे** — आपल्या टर्मिनलमध्ये `az login` चालवा.
3. **आवश्यक पर्यावरणीय चल सेट केले आहेत:**
   - `AZURE_AI_PROJECT_ENDPOINT` — आपला Azure AI Foundry प्रकल्प endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — आपल्याद्वारे तैनात केलेल्या मॉडेलचे नाव.

खालचा सेल आपल्याला आवश्यक असलेले Python पॅकेजेस इंस्टॉल करतो.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## आपला पहिला एजंट तयार करणे

एका एजंटला दोन गोष्टींची गरज असते:

- **सूचना** जी ते *कोण आहे* आणि *कसे वागायचे* ते सांगतात (एक सिस्टम प्रॉम्प्ट).
- **टूल्स** — Python फंक्शन्स ज्यांना `@tool` ने सजवलेले आहे जे एजंट माहिती मिळवण्यासाठी किंवा क्रिया करण्यासाठी कॉल करू शकतो.

खाली आपण एक सोपा टूल परिभाषित करतो जो लोकप्रिय सुट्टीच्या ठिकाणांची यादी परत करतो. वापरकर्ता प्रवास शिफारसी मागितल्यावर एजंट हा टूल वापरेल.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रीमिंग प्रतिसाद

अधिक संवादात्मक अनुभवासाठी आपण एजंटचा प्रतिसाद **स्ट्रीम** करू शकता. पूर्ण उत्तराची प्रतीक्षा करण्याऐवजी, एजंट तयार होत असलेल्या मजकुराचे तुकडे तुकडे देतो. हा पर्याय विशेषतः चॅट इंटरफेससाठी उपयुक्त आहे जेथे तुम्हाला परिणाम रिअल टाइममध्ये दर्शवायचा असतो.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

या धड्यात तुम्ही कसे शिकले:

- **प्रदाता तयार करा** जो `AzureAIProjectAgentProvider` द्वारे Azure AI Foundry Agent Service शी कनेक्ट होतो.
- **टूल परिभाषित करा** `@tool` डेकोरेटर वापरून जेणेकरून एजंट तुमच्या Python फंक्शन्सना कॉल करू शकेल.
- **एजंट चालवा** वापरकर्त्याचा संदेश देऊन आणि त्याची प्रतिक्रिया प्रिंट करा.
- **प्रतिसाद स्ट्रीम करा** वास्तविक वेळेतील आउटपुटसाठी.

पुढच्या धड्यात आपण एजंटिक फ्रेमवर्क्स अधिक सखोलपणे पाहणार आहोत आणि एजंट्सना अधिक सामर्थ्यशाली साधने आणि बहु-टप्प्यांचे विचार क्षमताही कशा देता येतील ते शिकणार आहोत.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI अनुवाद सेव्हिस [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेतील त्रुटी असू शकतात. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत म्हणून समजला पाहिजे. महत्त्वाच्या माहिती साठी व्यावसायिक मानवी अनुवाद करण्याची शिफारस केली जाते. या अनुवादाच्या वापरामुळे होणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थसंकल्पासाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
